# Chapter 09: Data Between Steps and Jobs: Outputs, Artifacts, Caching (notebook edition)

## Learning Objectives

- Verify outputs and artifacts across jobs
- Read cache hit and miss and bill them
- Find the output size limit

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

GHA_MODE = os.environ.get("GHA_MODE", "fixture")  # "fixture" | "live"
GHA_REPO = os.environ.get("GHA_REPO", "Friend09/practice_git_intro_to_github_actions")
print("mode:", GHA_MODE, "| repo:", GHA_REPO)

## 1. Outputs and the digest

Three runs of the data workflow; the version output is `0.1.<run_number>` and the artifact's content digest matched in the consumer job.

In [ ]:
import json
def load(n): return json.loads((ROOT/"fixtures"/n).read_text())
miss, hit, new = (load(f"data_ch09_{n}.json") for n in ("miss", "hit", "newkey"))
print([r["reports"]["consume"]["version"] for r in (miss, hit, new)])
assert [r["reports"]["consume"]["digest_match"] for r in (miss, hit, new)] == ["yes"] * 3

## 2. Cache hit vs miss

On a miss `cache-hit` is an empty string. We sum each run's cache-job steps and bill them.

In [ ]:
from intro_gha.cost import billed_minutes
tot = lambda r: sum(s["seconds"] for s in r["cache_job_steps"])
print(miss["reports"]["cache"]["cache_hit"], hit["reports"]["cache"]["cache_hit"])
print(tot(miss), tot(hit), tot(new), "->", billed_minutes(tot(miss)), billed_minutes(tot(hit)))
assert (tot(miss), tot(hit), tot(new)) == (17, 4, 14) and billed_minutes(tot(miss)) == billed_minutes(tot(hit)) == 1

## 3. Bad uploads

An empty path only warns by default; a duplicate name is a 409.

In [ ]:
ev = load("artifacts_cache_ch09.json")["artifact_events"]
print(ev["empty_default"]["level"], "|", ev["duplicate_name"]["message"][-60:])
assert ev["empty_default"]["level"] == "warning" and "409" in ev["duplicate_name"]["message"]

## 4. The output size limit

520,000 characters passed and 530,000 failed; the 1 MiB limit divided by 2 bytes per character is 524,288.

In [ ]:
lim = load("output_limits_ch09.json")
for p in lim["probes"]:
    print(p["job"], p["chars"], p["job_seconds"], "s", p["result"], "| fit:", p["quadratic_prediction_s"])
assert 520_000 * 2 <= 1_048_576 < 530_000 * 2
assert billed_minutes(128) == 3 and billed_minutes(215) == 4

## Takeaways & Next Steps

- Outputs are small strings; files go in artifacts; reusable files across runs go in a cache.
- `cache-hit` is empty on a miss.
- Next: Chapter 10, execution control.

## Chapter Link

Read: `learning_modules/chapter_09_data_outputs_artifacts_caching.md`